<a href="https://colab.research.google.com/github/swirita/Disease-Forecasting/blob/main/notebooks/01_preparing_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prepare Salmonellosis Weekly Data for Analysis

* **Original Dataset Source:** [CDC NNDSS Weekly Data](https://data.cdc.gov/NNDSS/NNDSS-Weekly-Data/x9gk-5huc/about_data)
* **Data Last Updated:** September 2, 2026
* **Group Members:** Husam, Siwar, and Abdallah

> This notebook will load and prepare the weekly Salmonellosis data published by the Centers for Disease Control. The dataset contains only **Salmonellosis (excluding Salmonella Typhi infection and Salmonella Paratyphi infection)**.

# INSTRUCTIONS

1. Download the filtered Salmonellosis dataset as a CSV file from the CDC NNDSS Weekly Data page
2. Compress the CSV file into a ZIP file named:

   `Salmonellosis_Weekly_Data.zip`

> **Important:** The data file is shared through Google Drive. The notebook and its code are saved in the groups GitHub repository.


In [2]:
# Imports
import pandas as pd
import zipfile
import os

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Load tha data from ZIP file
Zip_path = "/content/drive/MyDrive/AXSOSACADEMY/AXSOSACADEMY/06-AdvancedML/Week24/Data/Salmonellosis_Weekly_Data.csv.zip"
z = zipfile.ZipFile(Zip_path)
dfs = []
for file in z.namelist():
  if file.endswith('.csv'):
    df_year = pd.read_csv(z.open(file))
    dfs.append(df_year)
df = pd.concat(dfs, ignore_index=True)
z.close()


In [6]:
# Show tha Data
df.head()

,Reporting Area,Current MMWR Year,MMWR WEEK,Label,Current week,"Current week, flag",Previous 52 week Max,"Previous 52 weeks Max, flag",Cumulative YTD Current MMWR Year,"Cumulative YTD Current MMWR Year, flag",Cumulative YTD Previous MMWR Year,"Cumulative YTD Previous MMWR Year, flag",LOCATION1,LOCATION2,sort_order,geocode
0,US RESIDENTS,2022,1,Salmonellosis (excluding Salmonella Typhi infe...,96,-,"1,212",-,96,-,333,-,NaN,US RESIDENTS,20220105601,NaN
1,NEW ENGLAND,2022,1,Salmonellosis (excluding Salmonella Typhi infe...,2,-,82,-,2,-,24,-,NaN,NEW ENGLAND,20220105602,NaN
2,CONNECTICUT,2022,1,Salmonellosis (excluding Salmonella Typhi infe...,NaN,-,19,-,NaN,-,3,-,CONNECTICUT,NaN,20220105603,POINT (-72.738288 41.575155)
3,MAINE,2022,1,Salmonellosis (excluding Salmonella Typhi infe...,NaN,-,10,-,NaN,-,1,-,MAINE,NaN,20220105604,POINT (-69.06137 45.117911)
4,MASSACHUSETTS,2022,1,Salmonellosis (excluding Salmonella Typhi infe...,2,-,44,-,2,-,15,-,MASSACHUSETTS,NaN,20220105605,POINT (-71.481104 42.151077)


In [8]:
# Check the dataset size and basic information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17010 entries, 0 to 17009
Data columns (total 16 columns):
 #   Column                                   Non-Null Count  Dtype 
---  ------                                   --------------  ----- 
 0   Reporting Area                           17010 non-null  object
 1   Current MMWR Year                        17010 non-null  int64 
 2   MMWR WEEK                                17010 non-null  int64 
 3   Label                                    17010 non-null  object
 4   Current week                             9879 non-null   object
 5   Current week, flag                       13375 non-null  object
 6   Previous 52 week Max                     17005 non-null  object
 7   Previous 52 weeks Max, flag              10852 non-null  object
 8   Cumulative YTD Current MMWR Year         15293 non-null  object
 9   Cumulative YTD Current MMWR Year, flag   11439 non-null  object
 10  Cumulative YTD Previous MMWR Year        16177 non-null  o

## Missing Values

In [9]:
# Check missing values in each column
df.isna().sum()

,0
Reporting Area,0
Current MMWR Year,0
MMWR WEEK,0
Label,0
Current week,7131
"Current week, flag",3635
Previous 52 week Max,5
"Previous 52 weeks Max, flag",6158
Cumulative YTD Current MMWR Year,1717
"Cumulative YTD Current MMWR Year, flag",5571


## Check Current Week Flags

In [10]:
# Check the values and counts in the Current week flag column
df["Current week, flag"].value_counts(dropna=False)

,count
"Current week, flag",
-,13374
NaN,3635
U,1


In [12]:
# Compare missing Current week values with their flags
missing_with_dash = df[df["Current week"].isna() &(df["Current week, flag"] == "-")].shape[0]

missing_with_U = df[df["Current week"].isna() &(df["Current week, flag"] == "U")].shape[0]

missing_with_no_flag = df[df["Current week"].isna() &(df["Current week, flag"].isna())].shape[0]

print("Missing Current week with '-' flag:", missing_with_dash)
print("Missing Current week with 'U' flag:", missing_with_U)
print("Missing Current week with no flag:", missing_with_no_flag)

Missing Current week with '-' flag: 7130
Missing Current week with 'U' flag: 1
Missing Current week with no flag: 0


In [14]:
# Replace unreported cases with zero
df.loc[df["Current week"].isna() &(df["Current week, flag"] == "-"),"Current week"] = 0

# Remove the one row with unavailable data
df = df[df["Current week, flag"] != "U"].copy()

# Convert Current week to numeric
df["Current week"] = pd.to_numeric(df["Current week"].astype(str).str.replace(",", ""))

# Check the result
print("Number of rows:", df.shape[0])
print("Missing Current week values:", df["Current week"].isna().sum())
print("Current week data type:", df["Current week"].dtype)

Number of rows: 17009
Missing Current week values: 0
Current week data type: int64
